# Module 292 — Skills Assessment: IMDB Sentiment

Goal: predict **positive (1)** or **negative (0)** from a movie review, then optionally submit the trained model to HTB's authorized evaluator. This is a separate assessment notebook, not a modification of the earlier malware or network lessons. The course-supplied archive has `train.json` and `test.json`, each with 25,000 records containing `text` and `label`. Training and test stay separate.

A review becomes numbers through **TF–IDF**. `tf(t,d)` is read **term frequency of term t in document d**: how often a word occurs in one review. `df(t)` is **document frequency of term t**: how many training reviews contain it. With smoothed inverse document frequency, `idf(t)=log((1+N)/(1+df(t)))+1`; read **I-D-F of t equals the natural log of one plus the number of training documents N divided by one plus document frequency of t, then plus one**. A word in many reviews gets less weight than a more distinctive word. For N=9 and df=1, `idf=log(10/2)+1≈2.61`; for df=9, `idf=log(10/10)+1=1`. TF–IDF combines within-review frequency and across-training-review rarity, then normalizes each review vector. The vectorizer learns its vocabulary and IDF **only from train**, preventing test leakage.

A logistic classifier learns a weight for each text feature. `p(y=1|x)=1/(1+exp(−(w·x+b)))`; read **the probability of positive label y equals one given feature vector x is one divided by one plus e raised to the negative weighted sum w dot x plus bias b**. `w·x` (w dot x) adds each feature value times its learned weight. If the score `w·x+b` is zero, the probability is 0.5; if it is about 2, it is about 0.88. Here `argmax` is unnecessary: a probability at or above 0.5 maps to class 1. These weights are not human-readable proof of sentiment; they are a practical baseline learned from labeled examples.

The full pipeline keeps vectorization and classifier together inside the saved `joblib` file. **Never load an untrusted joblib file:** deserialization can execute code. The local test set is evaluated only after fitting. The HTB hidden evaluator is separate and may use a different distribution or threshold. No flag or target address is committed.

In [ ]:
import json
import os
from pathlib import Path
from zipfile import ZipFile
from collections import Counter

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline


## Load and check the HTB archive

Download the assessment archive from the HTB question. `IMDB_ASSESSMENT_ZIP` may override the default local Downloads path. We read the JSON members directly from the ZIP, so there is no large dataset extraction or accidental Git addition. The assertions establish the expected schema and binary labels before training; they are also helpful if HTB changes the download.

In [ ]:
DATA_ZIP = Path(os.environ.get('IMDB_ASSESSMENT_ZIP',
    Path.home() / 'Downloads' / '49df373a-992f-491b-8830-b827b5adf89d.zip'))
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f'Download the HTB assessment archive first: {DATA_ZIP}')
with ZipFile(DATA_ZIP) as archive:
    assert set(archive.namelist()) == {'train.json', 'test.json'}
    train_rows = json.loads(archive.read('train.json'))
    test_rows = json.loads(archive.read('test.json'))
def check_rows(rows, name):
    assert len(rows) == 25_000, f'Unexpected {name} row count'
    assert all(set(row) == {'text', 'label'} for row in rows)
    assert all(isinstance(row['text'], str) and row['text'] for row in rows)
    assert all(row['label'] in (0, 1) for row in rows)
    print(name, 'rows:', len(rows), 'labels:', dict(Counter(row['label'] for row in rows)))
check_rows(train_rows, 'Train')
check_rows(test_rows, 'Test')
train_texts = [row['text'] for row in train_rows]
train_labels = [row['label'] for row in train_rows]
test_texts = [row['text'] for row in test_rows]
test_labels = [row['label'] for row in test_rows]


## Fit one text-processing and classification pipeline

`TfidfVectorizer` learns vocabulary and IDF from training text. `ngram_range=(1,2)` includes single words and adjacent word pairs, so phrases like “not good” can differ from “good.” `max_features=50_000` bounds memory. `min_df=2` ignores terms seen in only one training review; `sublinear_tf=True` reduces the influence of repeated words by using `1+log(count)`. The sparse matrix stores only present features, rather than allocating 25,000 × 50,000 dense numbers. `LogisticRegression` then learns the text-feature weights. Its default L2 regularization discourages extreme weights. This is a course-aligned baseline, not an assertion that it matches an unspecified production text moderation system.

In [ ]:
sentiment_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50_000,
                               min_df=2, sublinear_tf=True, strip_accents='unicode')),
    ('classifier', LogisticRegression(solver='liblinear', max_iter=1000, random_state=1337)),
])
sentiment_model.fit(train_texts, train_labels)  # Learn only from train.json.
print('Learned text features:', len(sentiment_model['tfidf'].vocabulary_))
print('Classes:', sentiment_model.classes_.tolist())


## Evaluate once on the reserved test set

**Accuracy = correct predictions / all test predictions**, read **the number correct divided by the total**. A balanced dataset makes it easier to interpret than in a heavily skewed one, but errors still matter. The confusion matrix uses true labels as rows and predictions as columns: entry `Cᵢⱼ` (say **C sub i j**) counts actual class i predicted as class j. The per-class report shows precision and recall, so a strong overall number cannot hide every class-specific failure. Do not retune on this test result repeatedly; a separate validation split from training would be needed for parameter search.

In [ ]:
predicted = sentiment_model.predict(test_texts)
test_accuracy = accuracy_score(test_labels, predicted)
print(f'Local held-out accuracy: {test_accuracy:.4f}')
print('Confusion matrix (rows true 0,1; columns predicted 0,1):')
print(confusion_matrix(test_labels, predicted, labels=[0, 1]))
print(classification_report(test_labels, predicted, labels=[0, 1],
                            target_names=['negative', 'positive'], zero_division=0))


## Save and verify the trained artifact

HTB asks for a `skills_assessment.joblib` upload. Save the entire pipeline, not merely the logistic-regression object: the evaluator must apply the same learned vocabulary and IDF when it receives new raw review strings. The path defaults to Downloads outside Git and may be overridden with `IMDB_MODEL_FILE` for a writable location. We reload **only our own just-created file** and compare a few predictions before considering it ready for submission.

In [ ]:
MODEL_FILE = Path(os.environ.get('IMDB_MODEL_FILE',
    Path.home() / 'Downloads' / 'skills_assessment.joblib'))
joblib.dump(sentiment_model, MODEL_FILE, compress=3)
reloaded_model = joblib.load(MODEL_FILE)  # Trusted artifact created just above.
assert (reloaded_model.predict(test_texts[:5]) == predicted[:5]).all()
print('Verified model artifact:', MODEL_FILE, '| bytes:', MODEL_FILE.stat().st_size)


## Optional HTB submission

The Playground example uses `localhost:5000`; from a VPN-connected laptop use the **spawned HTB VM IP**, not localhost. The API expects an HTTP multipart POST with the file field named exactly `model`. Submission is disabled until the target is known. HTTP status alone does not prove the classifier passed: read the JSON response. Keep returned flags in the live output, not the committed notebook.

In [ ]:
import requests

RUN_HTB_SUBMISSION = False
HTB_VM_IP = ''  # Spawned authorized HTB target IP only; no scheme or port.
if RUN_HTB_SUBMISSION:
    if not HTB_VM_IP:
        raise ValueError('Set HTB_VM_IP to the spawned HTB VM address first.')
    endpoint = f'http://{HTB_VM_IP}:5000/api/upload'
    with MODEL_FILE.open('rb') as model_file:
        response = requests.post(endpoint, files={'model': model_file}, timeout=(10, 180))
    response.raise_for_status()
    print(json.dumps(response.json(), indent=2))
else:
    print('HTB submission disabled; no network request was made.')
